# 03 — Structured output: prefill + constrained decoding

Three layers make the small model's output reliable:

1. **No thinking**: Qwen3 otherwise emits a `<think>…</think>` block first.
2. **Response prefill**: the assistant turn is forced to start with `{"intent": "`.
3. **Constrained decoding**: a logits processor masks every token that cannot
   continue a valid intent name. Unknown intents become *impossible*, not just unlikely.

Then Pydantic validates the parameters (fail closed).

In [1]:
# --- setup: make the repo importable and quiet logs ---
import os, sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_OFFLINE", "1")  # models are already cached → fully local
logging.basicConfig(level=logging.WARNING)
print("repo:", ROOT)

repo: /Users/siavash/Projects/articles/narrow-ai-with-kv


In [2]:
from narrow_ai.settings import load_settings
from narrow_ai.models.model_registry import create_backend
from narrow_ai.assistant.orchestrator import IntentEngine
from narrow_ai.intents.registry import default_registry
from narrow_ai.models.prompt_templates import INTENT_PREFILL
s = load_settings(); s.model.backend = "mlx"
backend = create_backend(s)
engine = IntentEngine(s, backend, default_registry())
tok, model = backend.tokenizer, backend.model
print("allowed intents:", engine.allowed)

allowed intents: ['general_help', 'git_status', 'git_diff_summary', 'search_files', 'read_file', 'open_app', 'remember_preference', 'recall_preference', 'answer_question', 'clarify_request', 'unsupported']


## 1 · Why thinking must be off

In [3]:
from mlx_lm import generate
msgs = [{"role": "system", "content": engine.system_prompt}, {"role": "user", "content": "is my tree clean?"}]
with_thinking = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)  # Qwen3 default
text = generate(model, tok, prompt=with_thinking, max_tokens=80)
print(text[:400], "…" if len(text) > 400 else "")

<think>
Okay, the user is asking "is my tree clean?" Let me think. In Git, a clean working tree means there are no uncommitted changes. So the user wants to know if they've made any changes that need to be committed.

The relevant intent here is git_status. But wait, the user is asking about the tree being clean, which relates to the status of the working 


## 2 · Token-level view of the constraint

In [4]:
import mlx.core as mx
import numpy as np
from narrow_ai.models.constrained import IntentNameConstraint

def top(logits, k=5):
    lp = np.array((logits - mx.logsumexp(logits, keepdims=True))[0].astype(mx.float32))
    idx = lp.argsort()[::-1][:k]
    return ", ".join(f"{tok.decode([int(i)])!r}:{np.exp(lp[i]):.2f}" for i in idx)

def trace(utterance, steps=6):
    msgs = [{"role": "system", "content": engine.system_prompt}, {"role": "user", "content": utterance}]
    ids = backend.encode(backend.render(msgs, INTENT_PREFILL))
    con = IntentNameConstraint(tok, engine.allowed, n_prompt=len(ids))
    print(f"\n>>> {utterance!r}")
    for _ in range(steps):
        logits = model(mx.array(ids)[None])[:, -1, :]
        masked = con(mx.array(ids), logits)
        print(f"  free       : {top(logits)}")
        print(f"  constrained: {top(masked)}   allowed={len(con.allowed_ids(tok.decode(ids[con.n_prompt:])))}")
        nxt = int(mx.argmax(masked, axis=-1).item())
        ids.append(nxt)
        if con.done or tok.decode(ids[con.n_prompt:]).endswith('"'):
            break
    print("  → ", INTENT_PREFILL + tok.decode(ids[con.n_prompt:]))

trace("deploy my app to production")
trace("summarise what changed")


>>> 'deploy my app to production'
  free       : 'open':0.78, 'unsupported':0.17, 'clar':0.00, 'unknown':0.00, 'general':0.00
  constrained: 'open':0.82, 'unsupported':0.18, 'clar':0.00, 'general':0.00, 'cl':0.00   allowed=37
  free       : '_app':1.00, ' App':0.00, '_ap':0.00, '-app':0.00, '_APP':0.00
  constrained: '_app':1.00, '_ap':0.00, '_a':0.00, '_':0.00, ' pseud':0.00   allowed=4
  free       : '",':1.00, '”,':0.00, '","':0.00, "',":0.00, '$",':0.00
  constrained: '",':1.00, '","':0.00, '"},':0.00, '"",':0.00, '","",':0.00   allowed=62
  free       : ' "':1.00, ' ""':0.00, ' \\"':0.00, ' “':0.00, ' "+':0.00
  constrained: ' "':1.00, ' ""':0.00, ' \\"':0.00, ' “':0.00, ' "+':0.00   allowed=36883
  →  {"intent": "open_app", "

>>> 'summarise what changed'
  free       : 'git':1.00, ' git':0.00, '_git':0.00, '\tgit':0.00, '/git':0.00
  constrained: 'git':1.00, 'g':0.00, 'gi':0.00, 'gen':0.00, 'read':0.00   allowed=37
  free       : '_diff':1.00, ' Diff':0.00, 'Diff':0.00, '_delta

## 3 · Free vs constrained on tricky requests

In [5]:
import pandas as pd
from narrow_ai.intents.classifier import parse_intent_decision, IntentParseError
tricky = ["deploy to prod", "commit everything with message wip", "explain this stack trace",
          "rename main.py to app.py", "what's the time", "lint the project"]
rows = []
for q in tricky:
    msgs = [{"role": "system", "content": engine.system_prompt}, {"role": "user", "content": q}]
    row = {"request": q}
    for label, names in [("free", None), ("constrained", engine.allowed)]:
        g = backend.generate(msgs, prefill=INTENT_PREFILL, constrain_intents=names, max_tokens=64)
        try:
            row[label] = parse_intent_decision(g.text).intent.value
        except IntentParseError as e:
            row[label] = f"INVALID ({g.text[:40]}…)"
    rows.append(row)
pd.DataFrame(rows)

,request,free,constrained
0,deploy to prod,unsupported,unsupported
1,commit everything with message wip,git_status,git_status
2,explain this stack trace,answer_question,answer_question
3,rename main.py to app.py,read_file,read_file
4,what's the time,unsupported,unsupported
5,lint the project,unsupported,unsupported


## Try it

In [ ]:
import ipywidgets as W
from IPython.display import display
box = W.Text(value="ship it", description="request")
btn = W.Button(description="Decode", button_style="primary")
o = W.Output()
def go(_):
    with o:
        o.clear_output(); trace(box.value, steps=8)
btn.on_click(go); display(W.HBox([box, btn]), o)

Output()